# NARX V2

V2 search hyperparameter MLP trên order `NARX(1,5,2)`:

- 16 augmented features
- z-score train
- clip free-run
- chọn theo validation `FIT_sim`

V2 dùng `narx_pipeline.py` riêng, không dùng fit/simulate của ARX.


## 1. Import và cấu hình


In [1]:
from pathlib import Path
import json
import sys
import time

import numpy as np
import pandas as pd
from sklearn.neural_network import MLPRegressor

WORK_DIR = Path.cwd()
PROJECT_ROOT = WORK_DIR.parent if WORK_DIR.name == "NARX" else WORK_DIR
OUT_DIR = PROJECT_ROOT / "NARX"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from arx_pipeline import DataConfig, SplitConfig, load_or_generate_data, split_time_series
from narx_pipeline import (
    NarxConfig,
    build_augmented_df,
    fit_zscore_stats,
    apply_zscore,
    inverse_zscore_y,
    scaled_clip_bounds,
    build_narx_matrix,
    simulate_narx,
    compute_metrics,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

NA, NB, NK = 1, 5, 2
CLIP_QUANTILES = (0.01, 0.99)
RANDOM_STATE = 42

HIDDEN_OPTIONS = [(16,), (32,), (32, 16), (64, 32)]
ACTIVATIONS = ["tanh", "relu"]
ALPHAS = [1e-5, 1e-4, 1e-3, 1e-2]


## 2. Data và NARX matrix


In [2]:
DATA_CONFIG = DataConfig(
    csv_path=PROJECT_ROOT / "greenhouse_data.csv",
    generator_script_path=PROJECT_ROOT / "data_generator.py",
    force_regenerate_from_script=False,
    auto_save_generated_csv=True,
)
SPLIT_CONFIG = SplitConfig(train_ratio=0.60, val_ratio=0.20)

BASELINE_INPUT_COLS = ("Temperature", "Humidity", "Light", "Drip", "Mist", "Fan")
AUGMENTED_INPUT_COLS = (
    *BASELINE_INPUT_COLS,
    "Light_log",
    "Temp_x_Humi",
    "Temp_x_Light",
    "Humi_x_Light",
    "SP_Center",
    "SP_Width",
    "Month_sin",
    "Month_cos",
    "Season_sin",
    "Season_cos",
)

df_full, true_params, data_source = load_or_generate_data(DATA_CONFIG)
df_aug = build_augmented_df(df_full)
df_train, df_val, df_test = split_time_series(df_aug, SPLIT_CONFIG)

SCALE_COLS = ("Soil_Moisture", *AUGMENTED_INPUT_COLS)
scale_stats = fit_zscore_stats(df_train, SCALE_COLS)
clip_bounds_real, clip_bounds_scaled = scaled_clip_bounds(df_train, scale_stats, CLIP_QUANTILES)

df_train_z = apply_zscore(df_train, scale_stats)
df_val_z = apply_zscore(df_val, scale_stats)
df_test_z = apply_zscore(df_test, scale_stats)

CONFIG = NarxConfig(
    na=NA,
    nb=NB,
    nk=NK,
    input_cols=AUGMENTED_INPUT_COLS,
    simulation_clip=clip_bounds_scaled,
)

X_train, y_train = build_narx_matrix(df_train_z, CONFIG)
X_val, y_val = build_narx_matrix(df_val_z, CONFIG)
X_test, y_test = build_narx_matrix(df_test_z, CONFIG)

pd.Series({
    "data_source": data_source,
    "X_train": X_train.shape,
    "X_val": X_val.shape,
    "X_test": X_test.shape,
    "clip_real": clip_bounds_real,
    "clip_scaled": clip_bounds_scaled,
})


data_source                      CSV:greenhouse_data.csv
X_train                                      (63066, 81)
X_val                                        (21018, 81)
X_test                                       (21018, 81)
clip_real          (50.5465794049447, 64.61272806162447)
clip_scaled    (-2.0642032653307445, 2.1303300718324136)
dtype: object

## 3. Helper đánh giá


In [3]:
def one_step_metrics(model: MLPRegressor, x_mat: np.ndarray, y_vec: np.ndarray) -> dict[str, float]:
    pred = model.predict(x_mat)
    return compute_metrics(
        inverse_zscore_y(y_vec, scale_stats),
        inverse_zscore_y(pred, scale_stats),
        x_mat.shape[1],
    )


def sim_metrics(model: MLPRegressor, df_z: pd.DataFrame) -> dict[str, float]:
    y_pred, y_true = simulate_narx(df_z, model, CONFIG)
    return compute_metrics(
        inverse_zscore_y(y_true, scale_stats),
        inverse_zscore_y(y_pred, scale_stats),
        X_train.shape[1],
    )


def train_candidate(hidden: tuple[int, ...], activation: str, alpha: float, seed: int) -> tuple[MLPRegressor, dict]:
    model = MLPRegressor(
        hidden_layer_sizes=hidden,
        activation=activation,
        solver="adam",
        alpha=alpha,
        batch_size=512,
        learning_rate_init=1e-3,
        max_iter=250,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=15,
        random_state=seed,
        verbose=False,
    )
    start = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start
    val_1 = one_step_metrics(model, X_val, y_val)
    val_sim = sim_metrics(model, df_val_z)
    test_sim = sim_metrics(model, df_test_z)
    return model, {
        "hidden": list(hidden),
        "activation": activation,
        "alpha": float(alpha),
        "seed": int(seed),
        "n_iter": int(model.n_iter_),
        "loss": float(model.loss_),
        "best_validation_score": float(getattr(model, "best_validation_score_", np.nan)),
        "train_seconds": float(elapsed),
        "val_FIT_1step": val_1["FIT"],
        "val_RMSE_1step": val_1["RMSE"],
        "val_FIT_sim": val_sim["FIT"],
        "val_RMSE_sim": val_sim["RMSE"],
        "test_FIT_sim": test_sim["FIT"],
        "test_RMSE_sim": test_sim["RMSE"],
    }


## 4. Hyperparameter search


In [4]:
rows = []
models = {}
candidate_id = 0
total = len(HIDDEN_OPTIONS) * len(ACTIVATIONS) * len(ALPHAS)

for hidden in HIDDEN_OPTIONS:
    for activation in ACTIVATIONS:
        for alpha in ALPHAS:
            candidate_id += 1
            print(f"{candidate_id}/{total}: hidden={hidden}, activation={activation}, alpha={alpha}")
            model, row = train_candidate(hidden, activation, alpha, RANDOM_STATE)
            row["candidate_id"] = candidate_id
            rows.append(row)
            models[candidate_id] = model

search_df = pd.DataFrame(rows).sort_values(
    ["val_FIT_sim", "val_FIT_1step"],
    ascending=[False, False],
).reset_index(drop=True)

best_row = search_df.iloc[0].to_dict()
best_model = models[int(best_row["candidate_id"])]
search_df.round(4)


1/32: hidden=(16,), activation=tanh, alpha=1e-05


2/32: hidden=(16,), activation=tanh, alpha=0.0001


3/32: hidden=(16,), activation=tanh, alpha=0.001


4/32: hidden=(16,), activation=tanh, alpha=0.01


5/32: hidden=(16,), activation=relu, alpha=1e-05


...

29/32: hidden=(64, 32), activation=relu, alpha=1e-05


30/32: hidden=(64, 32), activation=relu, alpha=0.0001


31/32: hidden=(64, 32), activation=relu, alpha=0.001


32/32: hidden=(64, 32), activation=relu, alpha=0.01


,hidden,activation,alpha,seed,n_iter,loss,best_validation_score,train_seconds,val_FIT_1step,val_RMSE_1step,val_FIT_sim,val_RMSE_sim,test_FIT_sim,test_RMSE_sim,candidate_id
0,"[32, 16]",relu,0.0000,42,84,0.0046,0.9904,4.1979,88.1983,0.3543,65.9420,1.0225,53.9242,1.3422,21
1,"[32, 16]",relu,0.0001,42,64,0.0048,0.9902,3.2151,87.5637,0.3734,61.2199,1.1643,54.2491,1.3327,22
2,"[32, 16]",relu,0.0010,42,84,0.0047,0.9904,4.2364,87.8124,0.3659,54.7842,1.3575,25.6759,2.1650,23
3,[16],relu,0.0100,42,93,0.0049,0.9904,2.4799,85.8301,0.4254,22.4989,2.3268,-75.1191,5.1011,8
4,[16],relu,0.0001,42,93,0.0048,0.9903,2.4477,84.1165,0.4769,12.9970,2.6121,-92.3766,5.6038,6
5,[16],relu,0.0010,42,93,0.0048,0.9903,2.4589,85.4435,0.4370,12.3317,2.6321,-83.8104,5.3543,7
6,[16],tanh,0.0100,42,66,0.0054,0.9893,2.0002,80.3188,0.5909,1.3612,2.9615,19.3641,2.3489,4
7,[16],tanh,0.0010,42,94,0.0051,0.9897,2.8215,77.5702,0.6734,-1.0702,3.0345,16.3205,2.4375,3
8,[16],tanh,0.0001,42,94,0.0051,0.9897,2.8179,77.5124,0.6752,-1.2780,3.0407,15.9318,2.4489,2
9,[16],tanh,0.0000,42,94,0.0051,0.9897,2.8363,77.5077,0.6753,-1.2960,3.0413,15.8960,2.4499,1


## 5. Final metrics và so sánh


In [5]:
train_1 = one_step_metrics(best_model, X_train, y_train)
val_1 = one_step_metrics(best_model, X_val, y_val)
test_1 = one_step_metrics(best_model, X_test, y_test)
train_sim = sim_metrics(best_model, df_train_z)
val_sim = sim_metrics(best_model, df_val_z)
test_sim = sim_metrics(best_model, df_test_z)

metrics_df = pd.DataFrame([
    {"split": "Train", "FIT_1step": train_1["FIT"], "RMSE_1step": train_1["RMSE"], "FIT_sim": train_sim["FIT"], "RMSE_sim": train_sim["RMSE"], "Bias_sim": train_sim["Bias"]},
    {"split": "Validation", "FIT_1step": val_1["FIT"], "RMSE_1step": val_1["RMSE"], "FIT_sim": val_sim["FIT"], "RMSE_sim": val_sim["RMSE"], "Bias_sim": val_sim["Bias"]},
    {"split": "Test", "FIT_1step": test_1["FIT"], "RMSE_1step": test_1["RMSE"], "FIT_sim": test_sim["FIT"], "RMSE_sim": test_sim["RMSE"], "Bias_sim": test_sim["Bias"]},
]).round(4)

comparison_rows = []
for label, path in [
    ("ARX Search V1 best", PROJECT_ROOT / "ARX_Model_VersionSearch" / "arx_order_search_v1.json"),
    ("NARX V1", OUT_DIR / "narx_v1.json"),
]:
    if not path.exists():
        continue
    with path.open("r", encoding="utf-8") as f:
        artifact = json.load(f)
    if label.startswith("ARX"):
        best = artifact["best_by_validation"]
        comparison_rows.append({"model": label, "val_FIT_sim": best["val_FIT_sim"], "test_FIT_sim": best["test_FIT_sim"], "test_RMSE_sim": best["test_RMSE_sim"]})
    else:
        comparison_rows.append({"model": label, "val_FIT_sim": artifact["metrics"]["validation"]["fit_sim"], "test_FIT_sim": artifact["metrics"]["test"]["fit_sim"], "test_RMSE_sim": artifact["metrics"]["test"]["rmse_sim"]})

comparison_rows.append({"model": "NARX V2 best", "val_FIT_sim": val_sim["FIT"], "test_FIT_sim": test_sim["FIT"], "test_RMSE_sim": test_sim["RMSE"]})
comparison_df = pd.DataFrame(comparison_rows)
comparison_df["test_gain_vs_arx_search_best"] = comparison_df["test_FIT_sim"] - comparison_df.loc[0, "test_FIT_sim"]

display(metrics_df)
comparison_df.round(4)


,split,FIT_1step,RMSE_1step,FIT_sim,RMSE_sim,Bias_sim
0,Train,90.4515,0.3202,73.5212,0.8880,0.0765
1,Validation,88.1983,0.3543,65.9420,1.0225,-0.1434
2,Test,87.4776,0.3648,53.9242,1.3422,-0.1858


,model,val_FIT_sim,test_FIT_sim,test_RMSE_sim,test_gain_vs_arx_search_best
0,ARX Search V1 best,68.9559,66.8337,0.9661,0.0000
1,NARX V1,61.2199,54.2491,1.3327,-12.5847
2,NARX V2 best,65.9420,53.9242,1.3422,-12.9096


## 6. Lưu artifact


In [6]:
def json_ready(value):
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    if isinstance(value, np.ndarray):
        return json_ready(value.tolist())
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, Path):
        return str(value)
    return value


artifact = {
    "model_type": "NARX",
    "version": "narx_v2_mlp_hyperparameter_search",
    "order": {"na": NA, "nb": NB, "nk": NK},
    "input_cols": list(AUGMENTED_INPUT_COLS),
    "normalization": {"method": "zscore", "fit_on": "train", "stats": scale_stats},
    "simulation_clip": {"enabled": True, "bounds_real": list(clip_bounds_real), "bounds_scaled": list(clip_bounds_scaled)},
    "search_space": {"hidden_options": [list(v) for v in HIDDEN_OPTIONS], "activations": ACTIVATIONS, "alphas": ALPHAS},
    "best_candidate": best_row,
    "all_results": search_df.to_dict(orient="records"),
    "metrics": {
        "train": {"fit_1": train_1["FIT"], "fit_sim": train_sim["FIT"], "rmse_sim": train_sim["RMSE"]},
        "validation": {"fit_1": val_1["FIT"], "fit_sim": val_sim["FIT"], "rmse_sim": val_sim["RMSE"]},
        "test": {"fit_1": test_1["FIT"], "fit_sim": test_sim["FIT"], "rmse_sim": test_sim["RMSE"]},
    },
    "comparison": comparison_df.to_dict(orient="records"),
}

OUT_DIR.mkdir(exist_ok=True)
out_path = OUT_DIR / "narx_v2.json"
with out_path.open("w", encoding="utf-8") as f:
    json.dump(json_ready(artifact), f, indent=2)
    f.write("\n")
out_path


WindowsPath('C:/Users/minht/OneDrive/Desktop/ARX-Model/NARX/narx_v2.json')